# Validation of inverse + surrogate model wtih Ansys Q3D

reads-in file saved at the end of ```ml_21_train_keras_surrogate_defined_loss.ipynb```

In [1]:
from squadds import SQuADDS_DB
import pandas as pd
from squadds import Analyzer
import matplotlib.pyplot as plt
from squadds import AnsysSimulator
import numpy as np
import time

We don't actually care about using SQuADDS to get a device at the moment, we just use the code block below to get a "template" of the SQUaDDS style dataframe.

In [2]:
''' grab SQuADDS entry as a template for ML predicted designs ''' 
db = SQuADDS_DB()
db.select_system("qubit")
db.select_qubit("TransmonCross")
df = db.create_system_df()

analyzer = Analyzer(db)

# we are not actually looking for these Hamiltonian parameters... 
target_params={"qubit_frequency_GHz": 4, "anharmonicity_MHz": -200}

pred_df = analyzer.find_closest(target_params=target_params,
                                       num_top=1,
                                       metric="Euclidean",
                                       display=True)

''' read in ML results ''' 
ML_results = pd.read_csv("ml_data/Hamiltonian-based-inverse+surrogate_MLP_results.csv") # real in ML test results

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/SQuADDS/SQuADDS_DB/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/SQuADDS/SQuADDS_DB/6b4001554a05a484bcbb5faa7363f9c3097d5174/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/SQuADDS/SQuADDS_DB/resolve/6b4001554a05a484bcbb5faa7363f9c3097d5174/SQuADDS_DB.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/SQuADDS/SQuADDS_DB/SQuADDS/SQuADDS_DB.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/SQuADDS/SQuADDS_DB/revision/6b4001554a05a484bcbb5faa7363f9c3097d5174 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/SQuADDS/SQuADDS_DB/resolve/6b4001554a05a484bcbb5faa7363f9c3097d5174/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://

## Sweep through ML design results, simulate each design with Anysys Q3D, and save sim results
There are only three multi-valued parameters in the SQuADDS qubit-TransmonCross-cap_matrix dataset:
* connection_pads.readout.claw_length
* connection_pads.readout.ground_spacing
* cross_length

The model only predicts these three design parameters. We take a SQuADDS database entry and substitute in the predicted parameters from the model.

In [7]:
# dictionary to save results in 
results = pd.DataFrame({"Sample":[],
                        "pred_H_params":[],
                        "ref_H_params":[],
                        "simulation_time":[]})

um = 10**6 ## ML model is trained in SI units (m), convert back to µm  
samples_to_test = range(len(ML_results))
failed = []

In [8]:
for sample in samples_to_test:

    try:
        ''' current testing sample '''
        this_device = ML_results.iloc[sample]
    
        ''' get ML predicted design parameters '''
    
        # predicted device parameters
        pred_claw_length = str(this_device["pred_connection_pads.readout.claw_length"] * um)+'um'
        pred_ground_spacing = str(this_device["pred_connection_pads.readout.ground_spacing"] * um)+'um'
        pred_cross_length = str(this_device["pred_cross_length"] * um)+'um'
    
        ''' create our predicted design option for Qiskit Metal '''
        pred_df.design_options.iloc[0]["connection_pads"]["readout"]["claw_length"] = pred_claw_length
        pred_df.design_options.iloc[0]["connection_pads"]["readout"]["ground_spacing"] = pred_ground_spacing
        pred_df.design_options.iloc[0]["cross_length"] = pred_cross_length
        pred_device = pred_df.iloc[0]
    
        ''' simulate predicted design '''
        pred_ansys_simulator = AnsysSimulator(analyzer, pred_device)
        start_time = time.perf_counter()
        pred_ansys_results = pred_ansys_simulator.simulate(pred_device)
        end_time = time.perf_counter()
        sim_time = end_time - start_time
        pred_capacitance_matrix = pred_ansys_results["sim_results"]
        
        print("Predicted Hamiltonian parameters:")
        pred_Hamiltonian_params = pred_ansys_simulator.get_xmon_info(pred_ansys_results) # get Hamiltonian parameters
    
        ref_Hamiltonian_params = {"qubit_frequency_GHz":this_device.ref_qubit_frequency_GHz,"anharmonicity_MHz":this_device.ref_anharmonicity_MHz}
        
        results.loc[len(results)] = [sample, pred_Hamiltonian_params,ref_Hamiltonian_params,sim_time]
    except:
        failed.append(sample)

selected system: qubit
───────────────────────────────────────────────── Starting Simulation ──────────────────────────────────────────────────

═══ Simulation Plan ═══
System Type: Single Component
Total Simulations: 1
Simulation Types:
  1. LOM (Capacitance)
═══════════════════════

Simulation Parameters:
  Setup: {'auto_increase_solution_order': True, 'enabled': True, 'freq_ghz': 5.0, 'max_passes': 30, 
'min_converged_passes': 1, 'min_passes': 2, 'name': 'sweep_setup', 'percent_error': 0.1, 'percent_refinement': 30, 
'reuse_selected_design': False, 'reuse_setup': False, 'run': {'box_plus_buffer': True, 'components': array(['Q'], 
dtype=object), 'name': 'sweep_v2.0', 'open_terminations': array([array(['Q', 'readout'], dtype=object)], dtype=object)},
'save_fields': False, 'solution_order': 'High', 'solver_type': 'Iterative'}


 C:\Users\firas\miniconda3\envs\squadds041\Lib\site-packages\qiskit_metal\qgeometries\qgeometries_handler.py: 528
INFO 10:17AM [connect_project]: Connecting to Ansys Desktop API...
INFO 10:17AM [load_ansys_project]: 	Opened Ansys App
INFO 10:17AM [load_ansys_project]: 	Opened Ansys Desktop v2023.2.0
INFO 10:17AM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/firas/Documents/Ansoft/
	Project:   Project23
INFO 10:17AM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d1 [Solution type: Q3D]
INFO 10:17AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 10:17AM [connect]: 	Connected to project "Project23" and design "LOMv2.0_q3d1" 😀 

INFO 10:17AM [__del__]: Disconnected from Ansys HFSS


the parameters ['run'] are unsupported, so they have been ignored


INFO 10:17AM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d2 [Solution type: Q3D]
WARNING 10:17AM [connect_setup]: 	No design setup detected.
WARNING 10:17AM [connect_setup]: 	Creating Q3D default setup.
INFO 10:17AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 10:17AM [get_setup]: 	Opened setup `sweep_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 10:17AM [analyze]: Analyzing setup sweep_setup
INFO 10:18AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\2\tmpdf9bpunc.txt, C, , sweep_setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
 C:\Users\firas\miniconda3\envs\squadds041\Lib\site-packages\pyEPR\ansys.py: 1498
 C:\Users\firas\miniconda3\envs\squadds041\Lib\site-packages\pyEPR\ansys.py: 1505
INFO 10:18AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\2\tmph2ovqv7u.txt, C, , sweep_setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 5

Simulation Completed Successfully!
Predicted Hamiltonian parameters:
qubit anharmonicity = -151 MHz 
qubit frequency = 4.111 GHz
selected system: qubit
───────────────────────────────────────────────── Starting Simulation ──────────────────────────────────────────────────

═══ Simulation Plan ═══
System Type: Single Component
Total Simulations: 1
Simulation Types:
  1. LOM (Capacitance)
═══════════════════════

Simulation Parameters:
  Setup: {'auto_increase_solution_order': True, 'enabled': True, 'freq_ghz': 5.0, 'max_passes': 30, 
'min_converged_passes': 1, 'min_passes': 2, 'name': 'sweep_setup', 'percent_error': 0.1, 'percent_refinement': 30, 
'reuse_selected_design': False, 'reuse_setup': False, 'run': {'box_plus_buffer': True, 'components': array(['Q'], 
dtype=object), 'name': 'sweep_v2.0', 'open_terminations': array([array(['Q', 'readout'], dtype=object)], dtype=object)},
'save_fields': False, 'solution_order': 'High', 'solver_type': 'Iterative'}


...

 C:\Users\firas\miniconda3\envs\squadds041\Lib\site-packages\qiskit_metal\qgeometries\qgeometries_handler.py: 528
INFO 12:30PM [connect_project]: Connecting to Ansys Desktop API...
INFO 12:30PM [load_ansys_project]: 	Opened Ansys App
INFO 12:30PM [load_ansys_project]: 	Opened Ansys Desktop v2023.2.0
INFO 12:30PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/firas/Documents/Ansoft/
	Project:   Project23


the parameters ['run'] are unsupported, so they have been ignored


INFO 12:30PM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d78 [Solution type: Q3D]
INFO 12:30PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 12:30PM [connect]: 	Connected to project "Project23" and design "LOMv2.0_q3d78" 😀 

INFO 12:30PM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d79 [Solution type: Q3D]
WARNING 12:30PM [connect_setup]: 	No design setup detected.
WARNING 12:30PM [connect_setup]: 	Creating Q3D default setup.
INFO 12:30PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 12:30PM [get_setup]: 	Opened setup `sweep_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 12:30PM [analyze]: Analyzing setup sweep_setup
INFO 12:32PM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\2\tmpz6j1xhfk.txt, C, , sweep_setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
 C:\Users\firas\miniconda3\envs\squadds041\Lib\site-packages\pyE

Simulation Completed Successfully!
Predicted Hamiltonian parameters:
qubit anharmonicity = -145 MHz 
qubit frequency = 4.04 GHz
selected system: qubit
───────────────────────────────────────────────── Starting Simulation ──────────────────────────────────────────────────

═══ Simulation Plan ═══
System Type: Single Component
Total Simulations: 1
Simulation Types:
  1. LOM (Capacitance)
═══════════════════════

Simulation Parameters:
  Setup: {'auto_increase_solution_order': True, 'enabled': True, 'freq_ghz': 5.0, 'max_passes': 30, 
'min_converged_passes': 1, 'min_passes': 2, 'name': 'sweep_setup', 'percent_error': 0.1, 'percent_refinement': 30, 
'reuse_selected_design': False, 'reuse_setup': False, 'run': {'box_plus_buffer': True, 'components': array(['Q'], 
dtype=object), 'name': 'sweep_v2.0', 'open_terminations': array([array(['Q', 'readout'], dtype=object)], dtype=object)},
'save_fields': False, 'solution_order': 'High', 'solver_type': 'Iterative'}


 C:\Users\firas\miniconda3\envs\squadds041\Lib\site-packages\qiskit_metal\qgeometries\qgeometries_handler.py: 528
INFO 12:32PM [connect_project]: Connecting to Ansys Desktop API...
INFO 12:32PM [load_ansys_project]: 	Opened Ansys App
INFO 12:32PM [load_ansys_project]: 	Opened Ansys Desktop v2023.2.0
IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [11]:
results.to_csv("sim_data/Hamiltonian-based-inverse+surrogate_Ansys_results.csv",index=False)